# Introduction to Chilbolton Atmospheric Observatory

This notebook introduces the **Chilbolton Atmospheric Observatory (CAO)**, the instruments used in this project, and the structure of the datasets you will be analysing.

**Topics covered:**
1. The observatory — location and context
2. The instruments and variables
3. Dataset overview — files, time periods, and temporal resolution
4. A first look at the data — loading and plotting one variable from each instrument
5. Data availability — how complete are the records?

---
## 1. The Observatory

Chilbolton Atmospheric Observatory is a long-term atmospheric monitoring site operated by the **STFC RAL Space**, located near the village of Chilbolton in Hampshire, England.

| Property | Value |
|---|---|
| Full name | Chilbolton Atmospheric Observatory (CAO) |
| Location | Chilbolton, Hampshire, England |
| Latitude / Longitude | 51.145°N, 1.440°W |
| Altitude | 84 m above mean sea level |
| Project | Chilbolton Long Term Observations |

CAO sits within the broader **Chilbolton Observatory** site, which is most well known for its 25-metre steerable dish — one of the largest fully steerable radio telescopes in the world — used for radar meteorology and surveillance of objects in Low Earth Orbit (LEO). The surface meteorology instruments used in this project are located on the observatory grounds.

The site is in a rural location in the Test Valley, roughly 15 km south-west of Winchester. Its location in lowland southern England means it experiences a **temperate maritime climate**: mild, wet winters dominated by Atlantic westerlies, and drier, warmer summers — though rarely extreme.

---
## 2. Instruments and Variables

The datasets in this project come from four instruments sampling continuously at **10-second intervals**.

| Instrument | Manufacturer & Model | Variables | Dataset period |
|---|---|---|---|
| Cup anemometer & wind vane | Vector Instruments A100H & W200P | Wind speed (m s⁻¹), wind direction (°) | Apr 2014 – present |
| Temperature & humidity sensor | Vaisala HMP155A | Air temperature (K → °C), relative humidity (%) | Apr 2015 – present |
| Barometric pressure sensor | Vaisala PTB110 | Air pressure (hPa) | Apr 2019 – present |
| Drop counting raingauge | RAL | Precipitation amount (mm), precipitation rate (mm h⁻¹) | Apr 2014 – present |

All instruments record a 10-second mean. Data are stored in daily **NetCDF** files following the [CF-1.6](https://cfconventions.org/) and [NCAS-GENERAL-2.2.0](https://github.com/ncasuk/AMF_CVs) metadata conventions.

The drop counting raingauge works by directing rainfall through a calibrated orifice onto a funnel and counting individual drops as they fall through an optical sensor. Each drop corresponds to a known volume of water, allowing precipitation rate and accumulation to be recorded at high temporal resolution. The instrument is less susceptible to wind-induced undercatch than tipping-bucket designs and has no moving parts subject to mechanical wear.

Every variable has an associated **QC flag**:

| Flag value | Meaning |
|---|---|
| 0 | Not used |
| 1 | Good data |
| 2 | Bad data / measurement suspect |
| 3+ | Bad data — specific reason (instrument-dependent) |

For analysis you should always filter to **flag = 1** (good data only).


---
## 3. Dataset Overview

In [ ]:
from pathlib import Path
import netCDF4 as nc4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOTS = {
    'Wind':        '/data/wexp/cwalden/mean-winds',
    'Temperature & RH': '/data/wexp/cwalden/temperature-rh',
    'Pressure':    '/data/wexp/cwalden/pressure',
    'Precipitation': '/data/wexp/cwalden/precipitation'
}

print(f'{'Dataset':<22} {'Files':>6}  {'Start':>12}  {'End':>12}')
print('-' * 58)
for label, root in ROOTS.items():
    files = sorted(Path(root).rglob('*.nc'))
    with nc4.Dataset(str(files[0]))  as nc: t0 = getattr(nc, 'time_coverage_start', '?')[:10]
    with nc4.Dataset(str(files[-1])) as nc: t1 = getattr(nc, 'time_coverage_end',   '?')[:10]
    print(f'{label:<22} {len(files):>6}  {t0:>12}  {t1:>12}')

SyntaxError: f-string: expecting '}' (1064969911.py, line 14)

Each file contains one day of 10-second data. For a full day that is up to **8,640 records** per file.

The directory structure is organised by year:
```
/gws/ssde/j25a/chil_atmos/wx2026/
    mean-winds/
        2014/   ncas-anemometer-2_cao_20140401_mean-winds_v1.1.nc
               ncas-anemometer-2_cao_20140402_mean-winds_v1.1.nc  ...
        2015/   ...
        ...
    temperature-rh/
        2015/   ncas-temperature-rh-1_cao_20150415_surface-met_v1.1.nc ...
    pressure/
        2019/   ncas-pressure-1_cao_20190411_surface-met_v1.1.nc ...
```

The filename convention is:
```
<instrument-id>_<site-code>_<YYYYMMDD>_<product>_<version>.nc
```

---
## 4. A First Look at the Data

Let's load one week of data from each instrument and plot it to get a feel for what we are working with.

In [ ]:
def load_week(root, time_var, data_var, qc_var, qc_alt=None,
              year=2022, month=2, day_start=7, n_days=7):
    """Load n_days of 10-s data starting from year-month-day_start."""
    from datetime import date, timedelta
    target_dates = {
        (date(year, month, day_start) + timedelta(d)).strftime('%Y%m%d')
        for d in range(n_days)
    }
    files = [f for f in sorted(Path(root).rglob('*.nc'))
             if any(d in f.name for d in target_dates)]
    chunks = []
    for f in files:
        with nc4.Dataset(str(f)) as nc:
            unix = nc.variables[time_var][:].data.copy().astype(np.float64)
            vals = nc.variables[data_var][:].data.copy().astype(np.float64)
            qc_v = nc.variables.get(qc_var) or (nc.variables.get(qc_alt) if qc_alt else None)
            qc   = qc_v[:].data.copy().astype(np.int8) if qc_v is not None else np.ones(len(unix), dtype=np.int8)
        vals[(qc != 0) & (qc != 1)] = np.nan
        chunks.append(pd.DataFrame({'unix': unix, 'value': vals}))
    if not chunks:
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.concat(chunks, ignore_index=True).sort_values('unix').reset_index(drop=True)
    df['time'] = pd.to_datetime(df['unix'], unit='s', utc=True)
    return df.drop(columns='unix')


wind = load_week(ROOTS['Wind'], 'time', 'wind_speed', 'qc_flag_wind_speed')
temp = load_week(ROOTS['Temperature & RH'], 'time', 'air_temperature', 'qc_flag_air_temperature')
pres = load_week(ROOTS['Pressure'], 'time', 'air_pressure', 'qc_flag_air_pressure')

# Convert temperature from K to °C
temp['value'] = temp['value'] - 273.15

for label, df in [('Wind', wind), ('Temperature', temp), ('Pressure', pres)]:
    n_good = df['value'].notna().sum()
    print(f'{label:<15}: {len(df):,} records,  {n_good:,} good  '
          f'({100*n_good/max(len(df),1):.1f}%)')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(wind['time'], wind['value'], linewidth=0.5, color='steelblue')
axes[0].set_ylabel('Wind speed (m s⁻¹)')
axes[0].set_title('One week of surface meteorology at Chilbolton (7–13 Feb 2022)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(temp['time'], temp['value'], linewidth=0.5, color='tomato')
axes[1].set_ylabel('Temperature (°C)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(pres['time'], pres['value'], linewidth=0.5, color='mediumpurple')
axes[2].set_ylabel('Pressure (hPa)')
axes[2].grid(True, alpha=0.3)

axes[2].xaxis.set_major_locator(mdates.DayLocator())
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
axes[2].set_xlabel('Date (UTC)')

fig.tight_layout()
plt.show()

Even in a single week you can see the signatures of passing weather systems: pressure rises and falls as highs and lows move across the UK, temperature tracks the pressure changes and the diurnal (day/night) cycle, and wind speed spikes during frontal passages.

---
## 5. Data Availability

No instrument runs perfectly continuously. Gaps arise from power outages, instrument maintenance, data transmission failures, or QC rejection. It is important to know how complete a record is before drawing climatological conclusions.

We calculate the fraction of expected 10-second records that are present and flagged as good, for each calendar year.

In [ ]:
def annual_availability(root, data_var, qc_var, qc_alt=None, fill_value=None):
    """Return a Series of fraction-good per year."""
    files = sorted(Path(root).rglob('*.nc'))
    records = []   # (year, n_total, n_good)
    for f in files:
        with nc4.Dataset(str(f)) as nc:
            vals = nc.variables[data_var][:].data.copy().astype(np.float64)
            qc_v = nc.variables.get(qc_var) or (nc.variables.get(qc_alt) if qc_alt else None)
            qc   = qc_v[:].data.copy().astype(np.int8) if qc_v is not None else np.ones(len(vals), dtype=np.int8)
            t0   = getattr(nc, 'time_coverage_start', '')[:4]
        year  = int(t0) if t0.isdigit() else None
        if year is None:
            continue
        good = np.sum((qc == 0) | (qc == 1))
        records.append((year, len(vals), int(good)))
    df = pd.DataFrame(records, columns=['year', 'total', 'good'])
    annual = df.groupby('year')[['total','good']].sum()
    annual['availability'] = annual['good'] / annual['total'] * 100
    return annual['availability']


print('Computing availability (this may take a moment)...')
avail_wind = annual_availability(ROOTS['Wind'],        'wind_speed',      'qc_flag_wind_speed')
avail_temp = annual_availability(ROOTS['Temperature & RH'], 'air_temperature', 'qc_flag_air_temperature')
avail_pres = annual_availability(ROOTS['Pressure'],    'air_pressure',    'qc_flag_air_pressure')
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

width = 0.28
all_years = sorted(set(avail_wind.index) | set(avail_temp.index) | set(avail_pres.index))
x = np.arange(len(all_years))

def bar_vals(avail):
    return [avail.get(yr, np.nan) for yr in all_years]

ax.bar(x - width, bar_vals(avail_wind), width, label='Wind',        color='steelblue',    alpha=0.85)
ax.bar(x,         bar_vals(avail_temp), width, label='Temperature', color='tomato',        alpha=0.85)
ax.bar(x + width, bar_vals(avail_pres), width, label='Pressure',    color='mediumpurple',  alpha=0.85)

ax.axhline(90, linestyle='--', color='grey', linewidth=1, label='90% threshold')
ax.set_xticks(x)
ax.set_xticklabels(all_years, rotation=45)
ax.set_ylabel('Data availability (%)')
ax.set_ylim(0, 105)
ax.set_title('Annual data availability by instrument — Chilbolton Observatory')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

print('\nAnnual availability (%) per instrument:')
summary = pd.DataFrame({'Wind': avail_wind, 'Temperature': avail_temp, 'Pressure': avail_pres})
print(summary.round(1).to_string())

---
## Summary

| | |
|---|---|
| **Site** | Chilbolton Atmospheric Observatory, Hampshire (51.145°N, 1.440°W, 84 m) |
| **Operator** | NCAS — National Centre for Atmospheric Science |
| **Sampling rate** | 10 seconds (all instruments) |
| **File format** | NetCDF, CF-1.6 / NCAS-GENERAL-2.2.0 conventions |
| **QC convention** | Flag 1 = good, Flag ≥ 2 = bad — always filter before analysis |
| **Wind data** | Apr 2014 – present (Vector Instruments cup anemometer & wind vane) |
| **Temperature & RH** | Apr 2015 – present (Vaisala HMP155A) |
| **Pressure** | Apr 2019 – present (Vaisala PTB110) |

You are now ready to work through the analysis notebooks. Before starting each one, remember to:
1. Check data availability for the period you are analysing
2. Always apply QC flags before computing any statistics
3. Record the date range of your dataset in your report